═══════════════════════════════════════════════════════════════════
Paper 2 — AI-Driven Intervention Portfolio Optimisation for
          Industrial Emissions Reduction

Authors: Idris Alugo
Journal: (target)
Zenodo:  (DOI to be assigned)

Entry point: run this notebook top-to-bottom to reproduce
             all figures and tables in the paper.
═══════════════════════════════════════════════════════════════════

In [1]:
# ── Cell 0: Environment check ─────────────────────────────────────
import sys, importlib
print(f"Python {sys.version}")
required = ["numpy","pandas","sklearn","scipy","matplotlib"]
for pkg in required:
    try:    importlib.import_module(pkg); print(f"  ✓ {pkg}")
    except ImportError: print(f"  ✗ {pkg}  MISSING")

Python 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
  ✓ numpy
  ✓ pandas
  ✓ sklearn
  ✓ scipy
  ✓ matplotlib


In [2]:
# ── Cell 1: Imports & logging ─────────────────────────────────────
import logging, warnings, os
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

In [3]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))        # adjust if needed

In [4]:
from config import (
    DATA_PATH, ESG_PATH, INTERVENTION_PATH, FIGURES_DIR, OUTPUTS_DIR,
    K_CLUSTERS, CLUSTER_SEED, N_INIT,
    BUDGET_TOTAL, BUDGET_PER_FAC, RANDOM_SEED,
)
from src.clustering      import (build_facility_profile, run_kmeans,
                                  assign_tiers, elbow_silhouette_scan)
from src.interventions   import (load_interventions, filter_by_tier,
                                  compute_cost_effectiveness)
from src.optimisation    import (greedy_portfolio, lp_portfolio,
                                  portfolio_summary, lp_min_budget)
from src.visualization_p2 import (plot_figure6, plot_elbow_silhouette,
                                   plot_cost_effectiveness,
                                   plot_abatement_frontier)

In [5]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("paper2_pipeline")
np.random.seed(RANDOM_SEED)
logger.info("Imports complete.")

19:20:10 INFO     paper2_pipeline | Imports complete.


In [6]:
# ── Cell 2: Load Paper 1 outputs ──────────────────────────────────
# risk_df.csv is the output of Paper 1 (01_run_pipeline.ipynb cell 6)
# Columns: Facility, Prediction, Target_2030, Uncertainty_sigma,
#          P_i_percent, RiskLevel
risk_df = pd.read_csv(DATA_PATH)
logger.info("Paper 1 risk data loaded: %d facilities", len(risk_df))

19:20:11 INFO     paper2_pipeline | Paper 1 risk data loaded: 20 facilities


In [7]:
# Normalise column names from Paper 1 output to Paper 2 expected schema
risk_df = risk_df.rename(columns={
    "PredictionEnsemble":      "Prediction",
    "ProbMeetTargetEnsemble":  "P_i_percent",
    "RiskLevelEnsemble":       "RiskLevel",
    "Target2030":              "Target_2030",
    "UncertaintyEnsemble":     "Uncertainty_sigma",
})
logger.info("Paper 1 risk data loaded: %d facilities | columns: %s",
            len(risk_df), risk_df.columns.tolist())

19:20:11 INFO     paper2_pipeline | Paper 1 risk data loaded: 20 facilities | columns: ['Facility', 'Target_2030', 'ProbMeetTargetNHITS', 'ProbMeetTargetXGBoost', 'ProbMeetTargetBNN', 'P_i_percent', 'PredictionNHITS', 'PredictionXGBoost', 'PredictionBNN', 'Prediction', 'UncertaintyNHITS', 'UncertaintyXGBoost', 'UncertaintyBNN', 'Uncertainty_sigma', 'RiskLevel', 'TierLabel']


In [8]:
# Also load the raw emissions series for profile building
# (same esgdata.csv used in Paper 1)
raw_df = pd.read_csv(ESG_PATH, parse_dates=["Date"])
logger.info("Raw emissions data: %s", raw_df.shape)

19:20:11 INFO     paper2_pipeline | Raw emissions data: (2057, 11)


In [9]:
print(raw_df.columns.tolist())
print(raw_df.dtypes)

['Date', 'Region', 'Facility', 'Energy_MWh', 'Emission_Factor', 'Production', 'Emissions_tCO2', 'Waste_Kg', 'Renewable_percent', 'Transport_Emissions', 'PolicyWeight']
Date                   datetime64[us]
Region                            str
Facility                          str
Energy_MWh                    float64
Emission_Factor                   str
Production                    float64
Emissions_tCO2                float64
Waste_Kg                      float64
Renewable_percent             float64
Transport_Emissions             int64
PolicyWeight                  float64
dtype: object


In [10]:
# ── Filter to study facilities ────────────────────────────────────
STUDY_FACILITIES = [
    "F4", "F6", "F7", "F9", "F10", "F11", "F12", "F13",
    "F17", "F18", "F19", "F22", "F25", "F27", "F28", "F30",
    "F31", "F33", "F43", "F44",
]

risk_df = risk_df[risk_df["Facility"].isin(STUDY_FACILITIES)].reset_index(drop=True)
raw_df  = raw_df[raw_df["Facility"].isin(STUDY_FACILITIES)].reset_index(drop=True)

logger.info("Filtered to %d study facilities: %d risk rows, %d raw rows",
            len(STUDY_FACILITIES), len(risk_df), len(raw_df))

19:20:11 INFO     paper2_pipeline | Filtered to 20 study facilities: 20 risk rows, 883 raw rows


In [11]:
# ── Cell 3: Facility Profiling ────────────────────────────────────
profile = build_facility_profile(raw_df, target_col="Emissions_tCO2")
profile.to_csv(os.path.join(OUTPUTS_DIR, "facility_profiles.csv"))
logger.info("Facility profiles:\n%s", profile.describe().round(2).to_string())

19:20:11 INFO     src.clustering | Facility profiles built: 20 facilities × 6 features
19:20:11 INFO     paper2_pipeline | Facility profiles:
       mean_emissions  std_emissions  slope  mean_production  mean_energy  mean_renewable
count           20.00          20.00  20.00            20.00        20.00           20.00
mean           221.80         658.69 -13.81          4489.65      1577.43           39.14
std            282.90         627.99  12.47          5517.81      1900.05            0.22
min              4.12          25.12 -48.37           316.98       120.01           38.71
25%             79.99         310.00 -17.01          1331.67       487.17           39.11
50%            147.32         469.09 -10.04          2457.49       956.73           39.11
75%            223.50         847.17  -6.40          6147.86      1619.05           39.11
max           1132.98        2574.43  -0.49         25486.85      7364.20           40.00


In [12]:
import importlib, src.clustering, src.visualization_p2, src.optimisation
importlib.reload(src.clustering)
importlib.reload(src.visualization_p2)
importlib.reload(src.optimisation)
from src.clustering      import (build_facility_profile, run_kmeans,
                                  assign_tiers, elbow_silhouette_scan)
from src.visualization_p2 import (plot_figure6, plot_elbow_silhouette,
                                   plot_cost_effectiveness,
                                   plot_abatement_frontier)
from src.optimisation    import (greedy_portfolio, lp_portfolio,
                                  portfolio_summary, lp_min_budget)

In [13]:
# ── Cell 4: Elbow / Silhouette Scan (Appendix Fig S1) ────────────
scan_df = elbow_silhouette_scan(profile, k_range=range(2, 8),
                                 seed=CLUSTER_SEED, n_init=N_INIT)
print(scan_df.to_string(index=False))
plot_elbow_silhouette(scan_df, FIGURES_DIR)
scan_df.to_csv(os.path.join(OUTPUTS_DIR, "kmeans_scan.csv"), index=False)

 K   Inertia  Silhouette
 2 58.921079    0.615131
 3 40.495294    0.615700
 4 22.432835    0.588198
 5 14.806290    0.330805
 6 10.116479    0.295439
 7  5.930178    0.277250


19:20:13 INFO     src.visualization_p2 | Saved Figure S1 → c:\Users\Idris\Downloads\paper_2\figures\figureS1_elbow_silhouette.png


In [14]:
# ── Cell 5: K-means Clustering (K=3) ─────────────────────────────
cluster_result = run_kmeans(profile, k=K_CLUSTERS,
                             seed=CLUSTER_SEED, n_init=N_INIT)
tier_df = assign_tiers(cluster_result)
tier_df.to_csv(os.path.join(OUTPUTS_DIR, "facility_tiers.csv"))

# Drop TierLabel if already present (e.g. from a previous run saved in CSV)
if "TierLabel" in risk_df.columns:
    risk_df = risk_df.drop(columns=["TierLabel"])

# Join TierLabel into risk_df so all downstream outputs carry it
risk_df = risk_df.merge(
    tier_df[["TierLabel"]].reset_index().rename(columns={"index": "Facility"}),
    on="Facility", how="left"
)

# Persist updated risk_df (with TierLabel) back to disk
risk_df.to_csv(DATA_PATH, index=False)
logger.info("risk_df saved with TierLabel: %s", DATA_PATH)

19:20:13 INFO     src.clustering | K-means K=3  Silhouette=0.6157
19:20:13 INFO     src.clustering | Tier distribution (rank-based):
TierLabel
Low Emitters       10
Medium Emitters     8
High Emitters       2
19:20:13 INFO     paper2_pipeline | risk_df saved with TierLabel: c:\Users\Idris\Downloads\paper_2\data\risk_df.csv


In [15]:
print("\nTier distribution:")
print(tier_df["TierLabel"].value_counts().to_string())
print(f"\nGlobal Silhouette = {cluster_result['silhouette_global']:.4f}")


Tier distribution:
TierLabel
Low Emitters       10
Medium Emitters     8
High Emitters       2

Global Silhouette = 0.6157


In [16]:
# ── Cell 6: Figure 6 ─────────────────────────────────────────────
fig6_path = plot_figure6(cluster_result, tier_df, FIGURES_DIR, scan_df=scan_df)
print(f"Figure 6 saved → {fig6_path}")

19:20:13 INFO     src.visualization_p2 | Saved Figure 6 → c:\Users\Idris\Downloads\paper_2\figures\figure6_kmeans_clusters.png


Figure 6 saved → c:\Users\Idris\Downloads\paper_2\figures\figure6_kmeans_clusters.png


In [17]:
# ── Cell 7: Load Intervention Catalogue ──────────────────────────
interventions = load_interventions(INTERVENTION_PATH)
print(interventions[["InterventionID","Name","Category",
                      "Cost_EUR","AbatementRate"]].to_string(index=False))

19:20:13 INFO     src.interventions | Interventions loaded: 16 options across 5 categories


InterventionID                                  Name                 Category  Cost_EUR  AbatementRate
         EE-01                 LED Lighting Retrofit        Energy Efficiency     18000          0.040
         EE-02                HVAC Heat Pump Upgrade        Energy Efficiency     75000          0.090
         EE-03    Motor & Drive Efficiency (IE3/IE4)        Energy Efficiency     32000          0.060
         EE-04            Compressed Air Leak Repair        Energy Efficiency      8000          0.020
         EE-05            Thermal Insulation Upgrade        Energy Efficiency     22000          0.050
         RE-01         Rooftop Solar PV Installation         Renewable Energy    120000          0.120
         RE-02        Green Electricity Tariff / PPA         Renewable Energy      5000          0.180
         FS-01   Industrial Heat Pump (Process Heat)           Fuel Switching    180000          0.150
         FS-02            Biomass Boiler Replacement           Fuel Switc

In [18]:
# ── Cell 8: Build Per-Facility Candidate List ─────────────────────
# Use PredictionEnsemble (forward-looking) as facility baseline for
# abatement calculations, NOT historical mean_emissions.
# This ensures cost-per-tonne reflects the actual current trajectory,
# so large facilities with high predictions correctly dominate the LP.
risk_lookup = risk_df.set_index("Facility")["Prediction"]

candidates_list = []
for fac in tier_df.index:
    tier      = int(tier_df.loc[fac, "Tier"])
    baseline  = float(risk_lookup.loc[fac]) if fac in risk_lookup.index \
                else tier_df.loc[fac, "mean_emissions"]
    avail     = filter_by_tier(interventions, tier)
    costed    = compute_cost_effectiveness(avail, facility_baseline=baseline)
    costed["Facility"]  = fac
    costed["Tier"]      = tier
    costed["TierLabel"] = tier_df.loc[fac, "TierLabel"]
    candidates_list.append(costed)

In [19]:
all_candidates = pd.concat(candidates_list, ignore_index=True)
logger.info("Total candidate interventions: %d", len(all_candidates))
all_candidates.to_csv(os.path.join(OUTPUTS_DIR, "all_candidates.csv"), index=False)

19:20:13 INFO     paper2_pipeline | Total candidate interventions: 282


In [20]:
# ── Cell 9: Greedy Portfolio ──────────────────────────────────────
greedy_df = greedy_portfolio(all_candidates.copy(),
                              budget=BUDGET_TOTAL,
                              per_fac_cap=BUDGET_PER_FAC)
greedy_summary = portfolio_summary(greedy_df)
print("\nGreedy Portfolio Summary:")
print(greedy_summary.to_string(index=False))
greedy_summary.to_csv(os.path.join(OUTPUTS_DIR, "greedy_portfolio.csv"), index=False)

19:20:13 INFO     src.optimisation | Greedy portfolio: 10 interventions | abatement=422.8 tCO2 | cost=€199000



Greedy Portfolio Summary:
Facility  N_Interventions  Total_Cost_EUR  Total_Abatement  Avg_CostPerTonne
     F18                1            5000        13.215448        126.115029
     F22                1            5000        13.909700        119.820460
     F25                1            5000        19.644625         84.840849
     F28                1            5000        17.122581         97.337351
     F30                1            5000        21.092212         79.018107
     F43                3          147000       225.550833         36.859009
     F44                2           27000       112.234616         32.029466


In [21]:
# ── Cell 10: LP Portfolio ─────────────────────────────────────────
lp_df = lp_portfolio(all_candidates.copy(),
                      budget=BUDGET_TOTAL,
                      per_fac_cap=BUDGET_PER_FAC,
                      risk_df=risk_df)
lp_summary = portfolio_summary(lp_df)
print("\nLP Portfolio Summary:")
print(lp_summary.to_string(index=False))
lp_summary.to_csv(os.path.join(OUTPUTS_DIR, "lp_portfolio.csv"), index=False)

19:20:13 INFO     src.optimisation | LP portfolio: abatement=328.6 tCO2/month | cost=€200000



LP Portfolio Summary:
Facility  N_Interventions  Total_Cost_EUR  Total_Abatement  Avg_CostPerTonne   P_post  pred_post
     F10                1            5000         6.035248        276.155466 0.792145  27.493907
     F19                1            5000         1.593899       1045.653993 0.578078   7.261095
     F30                4           47000        32.810107        359.284046 0.755645  86.233058
     F43                7          125000       270.660999         72.676868 0.986667 354.437023


In [22]:
# ── Cell 11: Merge with Risk Data for Table 3 ─────────────────────
table3 = tier_df.merge(
    lp_summary.rename(columns={"Facility":"Facility"}),
    left_index=True, right_on="Facility", how="left"
).merge(
    risk_df[["Facility","Prediction","P_i_percent","RiskLevel"]],
    on="Facility", how="left"
).sort_values(["Tier","Prediction"])
print("\nTable 3 (LP + Risk):")
display_cols = ["Facility","TierLabel","Total_Cost_EUR",
                "Total_Abatement","RiskLevel","P_i_percent"]
if "P_post" in table3.columns:
    display_cols.append("P_post")
print(table3[display_cols].to_string(index=False))
table3.to_csv(os.path.join(OUTPUTS_DIR, "table3_portfolio_risk.csv"), index=False)


Table 3 (LP + Risk):
Facility       TierLabel  Total_Cost_EUR  Total_Abatement   RiskLevel  P_i_percent   P_post
      F4    Low Emitters             NaN              NaN   High Risk     0.260782      NaN
     F13    Low Emitters             NaN              NaN Medium Risk     0.662325      NaN
     F17    Low Emitters             NaN              NaN    Low Risk     0.890672      NaN
      F9    Low Emitters             NaN              NaN Medium Risk     0.389168      NaN
     F19    Low Emitters          5000.0         1.593899 Medium Risk     0.365731 0.578078
      F6    Low Emitters             NaN              NaN Medium Risk     0.549098      NaN
     F11    Low Emitters             NaN              NaN    Low Risk     0.907908      NaN
     F10    Low Emitters          5000.0         6.035248 Medium Risk     0.561562 0.792145
     F12    Low Emitters             NaN              NaN Medium Risk     0.615000      NaN
     F28    Low Emitters             NaN              NaN 

In [23]:
# ── Cell 12: Figure 8 – Cost-Effectiveness by Tier ───────────────
# Attach TierLabel to per-facility LP summary for the bar chart
ce_df = lp_summary.merge(
    tier_df[["TierLabel"]].reset_index(), on="Facility", how="left"
)
plot_cost_effectiveness(ce_df, FIGURES_DIR)

19:20:13 INFO     src.visualization_p2 | Saved Figure 8 → c:\Users\Idris\Downloads\paper_2\figures\figure8_cost_effectiveness.png


'c:\\Users\\Idris\\Downloads\\paper_2\\figures\\figure8_cost_effectiveness.png'

In [24]:
# ── Cell 13: Figure 9 – Abatement Frontier ───────────────────────
budgets    = np.linspace(100_000, 2_000_000, 20)
frontier   = []
for b in budgets:
    tmp = lp_portfolio(all_candidates.copy(), budget=b,
                       per_fac_cap=BUDGET_PER_FAC, risk_df=risk_df)
    abatement = (tmp["x_lp"] * tmp["AbatementVolume_tCO2"]).sum()
    frontier.append({"Budget_EUR": b, "Abatement_tCO2": abatement})
frontier_df = pd.DataFrame(frontier)
plot_abatement_frontier(frontier_df, FIGURES_DIR)
frontier_df.to_csv(os.path.join(OUTPUTS_DIR, "abatement_frontier.csv"), index=False)

19:20:14 INFO     src.optimisation | LP portfolio: abatement=263.6 tCO2/month | cost=€100000
19:20:14 INFO     src.optimisation | LP portfolio: abatement=328.6 tCO2/month | cost=€200000
19:20:14 INFO     src.optimisation | LP portfolio: abatement=349.6 tCO2/month | cost=€300000
19:20:14 INFO     src.optimisation | LP portfolio: abatement=358.0 tCO2/month | cost=€400000
19:20:14 INFO     src.optimisation | LP portfolio: abatement=361.7 tCO2/month | cost=€500000
19:20:14 INFO     src.optimisation | LP portfolio: abatement=363.5 tCO2/month | cost=€600000
19:20:14 INFO     src.optimisation | LP portfolio: abatement=364.8 tCO2/month | cost=€700000
19:20:14 INFO     src.optimisation | LP portfolio: abatement=365.4 tCO2/month | cost=€800000
19:20:14 INFO     src.optimisation | LP portfolio: abatement=365.5 tCO2/month | cost=€900000
19:20:14 INFO     src.optimisation | LP portfolio: abatement=365.5 tCO2/month | cost=€900000
19:20:14 INFO     src.optimisation | LP portfolio: abatement=365.5 tCO

In [25]:
# ── Cell 14: Minimum Budget to Achieve Portfolio P ≥ 0.80 ─────────
P_TARGET = 0.80

opt_budget, detail_df, portfolio_p = lp_min_budget(
    all_candidates.copy(), risk_df, p_target=P_TARGET
)

print("=" * 62)
print("MINIMUM BUDGET LP SOLUTION")
print("=" * 62)
print(f"OPTIMAL_BUDGET_EUR      = {opt_budget:>12,.0f}")
print(f"Portfolio P achieved    = {portfolio_p:.4f}  (target ≥ {P_TARGET})")
n_int = int((detail_df.groupby("Facility")["Alloc_EUR"].sum() > 1e-3).sum())
print(f"Facilities intervened   = {n_int} / {len(risk_df)}")
print()

alloc_df = detail_df[detail_df["Alloc_EUR"] > 1e-3].copy()
alloc_df["Alloc_EUR"] = alloc_df["Alloc_EUR"].round(0).astype(int)
print("Allocation detail (active interventions only):")
print(alloc_df[["Facility", "InterventionID", "InterventionName",
                 "Alloc_EUR", "Alloc_Abatement_tCO2",
                 "pred_pre", "pred_post", "p_pre", "p_post"]].to_string(index=False))

detail_df.to_csv(os.path.join(OUTPUTS_DIR, "lp_optimal_budget.csv"), index=False)
logger.info("Saved lp_optimal_budget.csv")

print()
print("=" * 62)
print("BUDGET SCENARIOS FOR RL EXPERIMENTS")
print("=" * 62)
print(f"  {'Scenario':<36} {'Budget (EUR)':>14}   {'Portfolio P':>11}")
print(f"  {'-'*36} {'-'*14}   {'-'*11}")

# Use opt_budget as the uncapped per-facility ceiling so the budget
# constraint (not the cap) is always the binding constraint.
for label, frac in [
    ("100% — full solution",    1.0),
    ("80%  — moderate scarcity", 0.8),
    ("70%  — tight",             0.7),
    ("60%  — severe scarcity",   0.6),
]:
    b = opt_budget * frac
    tmp = lp_portfolio(
        all_candidates.copy(),
        budget=b,
        per_fac_cap=b,          # non-binding per-facility cap
        risk_df=risk_df,
    )
    p_scen = float(tmp.drop_duplicates("Facility")["P_post"].mean())
    print(f"  {label:<36} {b:>14,.0f}   {p_scen:>11.4f}")

19:20:15 INFO     src.optimisation | Min-budget LP: optimal=€645708 | P_actual=0.731 | facilities=6/20
19:20:15 INFO     paper2_pipeline | Saved lp_optimal_budget.csv
19:20:15 INFO     src.optimisation | LP portfolio: abatement=629.2 tCO2/month | cost=€645708
19:20:15 INFO     src.optimisation | LP portfolio: abatement=553.5 tCO2/month | cost=€516567
19:20:15 INFO     src.optimisation | LP portfolio: abatement=515.7 tCO2/month | cost=€451996
19:20:15 INFO     src.optimisation | LP portfolio: abatement=474.6 tCO2/month | cost=€387425


MINIMUM BUDGET LP SOLUTION
OPTIMAL_BUDGET_EUR      =      645,708
Portfolio P achieved    = 0.7307  (target ≥ 0.8)
Facilities intervened   = 6 / 20

Allocation detail (active interventions only):
Facility InterventionID                     InterventionName  Alloc_EUR  Alloc_Abatement_tCO2   pred_pre  pred_post    p_pre   p_post
     F10          EE-01                LED Lighting Retrofit      18000              1.341166  33.529154  16.429286 0.561562 0.993798
     F10          EE-02               HVAC Heat Pump Upgrade      75000              3.017624  33.529154  16.429286 0.561562 0.993798
     F10          EE-03   Motor & Drive Efficiency (IE3/IE4)      32000              2.011749  33.529154  16.429286 0.561562 0.993798
     F10          EE-04           Compressed Air Leak Repair       8000              0.670583  33.529154  16.429286 0.561562 0.993798
     F10          EE-05           Thermal Insulation Upgrade      22000              1.676458  33.529154  16.429286 0.561562 0.993798


In [26]:
logger.info("Pipeline complete. All figures in %s", FIGURES_DIR)
logger.info("All tables  in %s", OUTPUTS_DIR)

19:20:15 INFO     paper2_pipeline | Pipeline complete. All figures in c:\Users\Idris\Downloads\paper_2\figures
19:20:15 INFO     paper2_pipeline | All tables  in c:\Users\Idris\Downloads\paper_2\outputs


In [27]:
# ── Clean summary of the min-budget LP solution ───────────────────
# detail_df is already in memory from Cell 14; no need to re-read disk.

# Facility-level summary (deduplicate – p_post is the same for all
# interventions belonging to the same facility)
fac_sum = detail_df.groupby("Facility").agg(
    total_cost=("Alloc_EUR",            "sum"),
    pred_pre  =("pred_pre",             "first"),
    pred_post =("pred_post",            "first"),
    p_pre     =("p_pre",                "first"),
    p_post    =("p_post",               "first"),
).reset_index()

print(f"{'Facility':<8} {'Cost':>10} {'pred_pre':>10} {'pred_post':>10} "
      f"{'P_pre':>7} {'P_post':>7} {'ΔP':>7}")
print("-" * 65)
for _, r in fac_sum.sort_values("total_cost", ascending=False).iterrows():
    print(f"{r['Facility']:<8} {r['total_cost']:>10,.0f} {r['pred_pre']:>10.1f} "
          f"{r['pred_post']:>10.1f} {r['p_pre']:>7.3f} {r['p_post']:>7.3f} "
          f"{r['p_post'] - r['p_pre']:>+7.3f}")

print(f"\nTotal budget used:     {fac_sum['total_cost'].sum():>10,.0f} EUR")
print(f"Portfolio P BEFORE:    {fac_sum['p_pre'].mean():>10.4f}")
print(f"Portfolio P AFTER LP:  {fac_sum['p_post'].mean():>10.4f}")
print(f"LP improvement:        {fac_sum['p_post'].mean() - fac_sum['p_pre'].mean():>+10.4f}")
print(f"\nFacilities with gap>0 that got budget:")
gap_facs = fac_sum[fac_sum["pred_post"] < fac_sum["pred_pre"]]
print(gap_facs[["Facility", "total_cost", "p_pre", "p_post"]].to_string(index=False))

Facility       Cost   pred_pre  pred_post   P_pre  P_post      ΔP
-----------------------------------------------------------------
F43         213,708      644.4      306.0   0.346   0.996  +0.650
F10         200,000       33.5       16.4   0.562   0.994  +0.432
F30          97,000      117.2       72.7   0.430   0.861  +0.430
F9           65,000        7.8        5.3   0.389   0.720  +0.331
F19          65,000        8.9        6.0   0.366   0.736  +0.370
F4            5,000        0.4        0.3   0.261   0.434  +0.174
F11               0       14.6       14.6   0.908   0.885  -0.023
F12               0       44.2       44.2   0.615   0.612  -0.003
F13               0        2.5        2.5   0.662   0.638  -0.024
F22               0       77.3       77.3   0.568   0.504  -0.063
F18               0       73.4       73.4   0.854   0.606  -0.248
F17               0        4.7        4.7   0.891   0.921  +0.031
F28               0       95.1       95.1   0.517   0.573  +0.056
F27       